# Feature engineering for residual load forecasting

Implements [`.claude/specs/05-feature-engineering.md`](../../.claude/specs/05-feature-engineering.md).

The project needs a lean, leakage-safe feature set to forecast `residual_load` one day ahead.
Per [CLAUDE.md](../../CLAUDE.md), generation features are limited to wind + solar, and SMARD's
own `fc_residual_load` is the benchmark our model has to beat — so it is never used as a model
input, only as the comparison target scored separately
(per [`04-forecast-metrics.md`](../../.claude/specs/04-forecast-metrics.md)).

## What this notebook produces

- The **forecast-origin contract**: for a target day `D`, a feature may only use
  `Forecast Wind + Solar` for `D` itself, actual history strictly before `D`'s first hour, and
  calendar facts about `D`.
- A fixed set of **12 features**, grouped by daily / weekly / seasonal pattern and by type
  (calendar, cyclical encoding, lag, rolling statistic, capacity-normalised ratio,
  forecast-derived).
- Documentation of the **2 dropped candidates** and the **3 explicitly excluded** columns, with
  reasons.
- An **origin-safety self-check**.
- One exported file: `data/features/residual_load_features.csv`.

## Not in this notebook

- Fitting any model, choosing a train/test split, or MLflow logging.
- Any feature from `Forecast Grid Load` or `Forecast Residual Load`.
- The extended families in `Hari_Gridstress_feature_engineering_baselines_metrics.ipynb`
  (spectral state, harmonic regression, intervention clocks) — parked in
  [`05.1-spectral-state.md`](../../.claude/specs/05.1-spectral-state.md), not run here.
- Joining risk labels — left to the modelling spec.

## 1 Setup

Same setup as [`team-EDA.ipynb`](../01_eda/team-EDA.ipynb) §1, as reused by
[`risk-definition.ipynb`](../03_risk_classification/risk-definition.ipynb) §1 and
[`forecast-metrics-claude.ipynb`](../02_forecast_metrics/forecast-metrics-claude.ipynb) §1.
Inherited, not re-derived:

- the data-directory resolver (walks **upward** from the working directory, so the notebook runs
  from any folder in the repo)
- loading, renaming, the German-CSV float conversion and the dtype asserts
- `time_series`, `SERIES`, `DERIVED`, `YEARS`, the season mapping

`data/` is **gitignored**, so `data/smard.csv` does not come with a clone. Regenerate it with
`notebooks/API-connection.ipynb`.

In [ ]:
from pathlib import Path

import holidays
import numpy as np
import pandas as pd

# Walk up from the working directory to the first parent holding a `data/` folder
DATA_DIR = next(
    (p / "data" for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(
        f"no data/ directory found in {Path.cwd()} or any parent — start the kernel inside the "
        "repository, then re-run."
    )
DATA = DATA_DIR / "smard.csv"
FORECAST_ERRORS = DATA_DIR / "metrics" / "smard_forecast_errors_hourly.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone — "
        "regenerate it by running notebooks/API-connection.ipynb top to bottom."
    )
if not FORECAST_ERRORS.exists():
    raise FileNotFoundError(
        f"{FORECAST_ERRORS} not found. Regenerate it by running "
        "notebooks/02_forecast_metrics/forecast-metrics-claude.ipynb top to bottom (spec 04)."
    )

print(f"Data directory: {DATA_DIR}")

### 1.1 Load and prepare

The dtype assertion guards against the German Excel-CSV conversion silently leaving a column as
text. Everything after this cell uses `time_series`.

In [ ]:
# The CSV headers exactly as notebooks/API-connection.ipynb writes them.
COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid Load": "fc_grid_load",
    "Forecast Residual Load": "fc_residual_load",
    "Capacity Wind Offshore": "cap_wind_off",
    "Capacity Wind Onshore": "cap_wind_on",
    "Capacity Solar": "cap_solar",
}

raw = pd.read_csv(DATA, delimiter=";", encoding="utf-8-sig")

assert set(raw.columns) == {"timestamp"} | set(COLUMNS), (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ ({'timestamp'} | set(COLUMNS)))}"
)

raw = raw.rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")

for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)

time_series = raw.set_index("timestamp").sort_index()
del raw  # the flat frame does not outlive the loading cell

assert all(
    pd.api.types.is_float_dtype(time_series[c]) for c in COLUMNS.values()
), time_series.dtypes

LOADED = {
    "rows": len(time_series),
    "start": time_series.index.min(),
    "end": time_series.index.max(),
}

print(f"shape           : {time_series.shape[0]:,} rows x {time_series.shape[1]} columns")
print(f"index           : {time_series.index.min()}  ->  {time_series.index.max()}")
print(
    f"index monotonic : {time_series.index.is_monotonic_increasing}, "
    f"unique: {time_series.index.is_unique}"
)
time_series.head(3)

### 1.2 Engineered columns

`YEARS` is computed from the loaded data and is the only permitted source of year information.

In [ ]:
SERIES = [
    "wind_off", "wind_on", "solar", "grid_load", "residual_load",
    "fc_gen_wind_solar", "fc_grid_load", "fc_residual_load",
    "cap_wind_off", "cap_wind_on", "cap_solar",
]

# Meteorological seasons, with December assigned to the FOLLOWING year's winter.
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

time_series["renewables"] = time_series[["wind_on", "wind_off", "solar"]].sum(axis=1)
time_series["year"] = time_series.index.year
time_series["month"] = time_series.index.month
time_series["hour"] = time_series.index.hour
time_series["dow"] = time_series.index.dayofweek
time_series["is_weekend"] = time_series.index.dayofweek >= 5
time_series["date"] = time_series.index.date
time_series["season"] = pd.Categorical(
    time_series.index.month.map(SEASON_OF_MONTH), categories=SEASON_ORDER, ordered=True
)
time_series["season_year"] = time_series.index.year + (time_series.index.month == 12)

# Outputs True on the row FOLLOWING a gap. The first row is False (NaT comparison), not NaN.
time_series["spans_gap"] = time_series.index.to_series().diff() > pd.Timedelta("1h")

DERIVED = [
    "renewables", "year", "month", "hour", "dow", "is_weekend",
    "date", "season", "season_year", "spans_gap",
]

assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

YEARS = sorted(int(y) for y in time_series["year"].unique())

print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {time_series.shape[1]} columns")
print(f"YEARS = {YEARS}")

### 1.3 The forecast-origin contract (origin safety)

For a target hour `t` on target day `D`, a feature may only read:

1. **calendar facts about `D`** — known infinitely far in advance (hour, day of week, day of
   year, weekend/holiday flag, season);
2. **actual history strictly before `D`'s first hour** — any lag or rolling window must end
   before `D` starts;
3. **`Forecast Wind + Solar` for `D` itself** — the one genuine day-ahead forecast in scope,
   published by 18:00 on `D−1` (per `04-forecast-metrics.md`), so it is legitimately known at
   origin.

Nothing from `D`'s actual `grid_load`, `renewables` or `residual_load` may appear as an input —
those are the outcome, not a feature. This rule is checked explicitly in §5.

### 1.4 Initial self-check

Structural only, and deliberately free of any hardcoded row count or date bound: the record's
extent is expected to change.

In [ ]:
assert all(pd.api.types.is_float_dtype(time_series[c]) for c in SERIES), time_series[SERIES].dtypes
assert time_series.index.is_monotonic_increasing, "index is not sorted"
assert time_series.index.is_unique, "index has duplicate timestamps"
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert YEARS == sorted(int(y) for y in time_series["year"].unique())

print("setup self-check passed")
print(f"  {len(SERIES)} float series, index sorted and unique")
print(f"  columns == SERIES + DERIVED ({len(SERIES) + len(DERIVED)} columns)")
print(f"  {LOADED['rows']:,} rows, {LOADED['start']} -> {LOADED['end']}, years {YEARS}")

## 2 Calendar / cyclical features

These describe facts about the target hour/day itself — never a measurement — so they carry
zero leakage risk under the origin-safety contract (§1.3):

| Feature | Definition |
|---|---|
| `hour_sin`, `hour_cos` | `sin/cos(2π·hour/24)` |
| `dow_sin`, `dow_cos` | `sin/cos(2π·dow/7)` |
| `doy_sin`, `doy_cos` | `sin/cos(2π·day_of_year/365.25)` |
| `is_weekend` | from `DERIVED["is_weekend"]` |
| `is_holiday` | `holidays.country_holidays("DE")` flag |
| `season` | from `DERIVED["season"]` |

`features` is built as a **separate DataFrame** on `time_series.index` (Behaviour 2), so
`time_series` keeps exactly `SERIES + DERIVED`.

In [ ]:
features = pd.DataFrame(index=time_series.index)

hour = time_series["hour"].to_numpy()
dow = time_series["dow"].to_numpy()
doy = time_series.index.dayofyear.to_numpy()

features["hour_sin"] = np.sin(2 * np.pi * hour / 24)
features["hour_cos"] = np.cos(2 * np.pi * hour / 24)
features["dow_sin"] = np.sin(2 * np.pi * dow / 7)
features["dow_cos"] = np.cos(2 * np.pi * dow / 7)
features["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
features["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)

features["is_weekend"] = time_series["is_weekend"].astype(int)

de_holidays = holidays.country_holidays("DE")
features["is_holiday"] = time_series["date"].map(lambda d: d in de_holidays).astype(int)

features["season"] = time_series["season"]

assert features.index.equals(time_series.index)
assert list(time_series.columns) == SERIES + DERIVED, "time_series must not gain feature columns"

print(f"features so far: {features.shape[0]:,} rows x {features.shape[1]} columns")
print(f"columns: {list(features.columns)}")
features.head(3)

**Sanity check.** `is_holiday` should mark a small, plausible share of hours (roughly one in
a few hundred, since Germany has about 9 nationwide holidays a year); `hour_sin`/`hour_cos`
should trace a full circle over 24 hours. Checked below.

In [ ]:
holiday_share = features["is_holiday"].mean()
weekend_share = features["is_weekend"].mean()
n_holiday_days = time_series.loc[features["is_holiday"] == 1, "date"].nunique()

print(f"is_holiday share : {holiday_share:.4%}  ({n_holiday_days} distinct holiday dates)")
print(f"is_weekend share  : {weekend_share:.4%}")
print(f"hour_sin/cos range: [{features['hour_sin'].min():.3f}, {features['hour_sin'].max():.3f}] / "
      f"[{features['hour_cos'].min():.3f}, {features['hour_cos'].max():.3f}]")
print(f"season categories : {features['season'].cat.categories.tolist()}")

assert 0 < holiday_share < 0.05, "holiday share looks implausible"
assert np.isclose(weekend_share, 2 / 7, atol=0.01), "weekend share should be close to 2/7"

### 2.1 Visual check: do the cyclical encodings trace a correct circle?

Each panel plots `(cos, sin)` for one cycle, coloured by the underlying ordinal value (sequential colourmap, since it is a magnitude, not a category). `hour` and `dow` are small integers, so each panel shows a **finite set of discrete, evenly spaced points** (24 and 7 respectively) rather than a continuous curve — that is expected, not a bug. `doy` ranges over 365/366 values and should trace a visibly continuous circle. In every panel, the two ends of the cycle (hour 23/hour 0, day 6/day 0, day 365-366/day 1) should sit immediately next to each other with no gap and no double-counted point.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

panels = [
    ("hour_cos", "hour_sin", time_series["hour"], "Hour of day (0–23)", axes[0]),
    ("dow_cos", "dow_sin", time_series["dow"], "Day of week (0=Mon…6=Sun)", axes[1]),
    ("doy_cos", "doy_sin", time_series.index.dayofyear, "Day of year (1–366)", axes[2]),
]

for x_col, y_col, color_values, title, ax in panels:
    scatter = ax.scatter(
        features[x_col], features[y_col],
        c=color_values, cmap="viridis", s=8, alpha=0.5, linewidths=0,
    )
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_aspect("equal")
    ax.set_xlim(-1.15, 1.15)
    ax.set_ylim(-1.15, 1.15)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    fig.colorbar(scatter, ax=ax, fraction=0.046, pad=0.06)

fig.suptitle("Cyclical calendar encodings — checking for a full, evenly spaced circle", y=1.02)
fig.tight_layout()
plt.show()

**Reading the panels.** `hour_sin/cos` and `dow_sin/cos` each show their full set of discrete points (24 and 7) evenly spaced around the circle, exactly as `hour/24` and `dow/7` should place them — hour 23 sits directly next to hour 0, day 6 next to day 0. `doy_sin/cos` traces a visibly continuous, closed circle with no gap: a fixed `365.25`-day period means day 366 of a leap year lands almost exactly on day 1 rather than leaving a visible break, and the ~0.25-day annual drift this introduces is far smaller than the plot's resolution. All three encodings are correctly phased and periodic; no adjustment is needed.

## 3 Lag and rolling features

Unlike the calendar features, these read actual measurements, so the origin-safety contract
(§1.3) is load-bearing here (Behaviour 4):

| Feature | Definition | How the cutoff is enforced |
|---|---|---|
| `residual_load_lag_24h` | actual `residual_load` 24h before target hour `t` | wall-clock shift by exactly 24h always lands one calendar day earlier, strictly before `t`'s day |
| `residual_load_lag_168h` | actual `residual_load` 168h (7 days) before `t` | same mechanism, 168h |
| `vre_capacity_factor_lag_24h` | `(wind_off+wind_on+solar) / (cap_wind_off+cap_wind_on+cap_solar)`, 24h before `t` | ratio computed first, then the same 24h wall-clock shift |
| `residual_load_roll_mean_72h` | trailing 72h mean, **one value per calendar day**, anchored at that day's first hour | `.rolling("72h", closed="left")` evaluated **only** at each day's midnight (excludes the midnight row itself), then broadcast to all hours of that day |
| `residual_load_roll_std_168h` | trailing 7-day std dev, same anchoring | same mechanism, 168h |

**Why the two rolling features are anchored at midnight, not per-hour.** A per-hour trailing
window (`.rolling(...).shift(1)`) would use data from *earlier the same day* for an evening
hour — information that would not actually exist yet at a `D−1` noon forecast origin. Anchoring
the window at `D`'s first hour and broadcasting to every hour of `D` means the whole day shares
one honestly-timed snapshot, matching how a real day-ahead forecast is actually produced (once
per origin, not re-computed hour by hour through the target day). The two lag features do not
need this treatment: a 24h/168h lag from *any* hour of day `D` always lands in a fully earlier
calendar day, so it is safe per-hour without anchoring.

In [ ]:
def wall_clock_lag(series: pd.Series, hours: int) -> pd.Series:
    """`series` value exactly `hours` of wall-clock time earlier, aligned back onto
    `series.index`. Local-time shift (not a row-count shift), consistent with the project's
    "same local hour, N days earlier" persistence convention.
    """
    shifted = series.copy()
    shifted.index = shifted.index + pd.Timedelta(hours=hours)
    return shifted.reindex(series.index)


features["residual_load_lag_24h"] = wall_clock_lag(time_series["residual_load"], 24)
features["residual_load_lag_168h"] = wall_clock_lag(time_series["residual_load"], 168)

vre_capacity = time_series["cap_wind_off"] + time_series["cap_wind_on"] + time_series["cap_solar"]
vre_capacity_factor = time_series["renewables"] / vre_capacity
features["vre_capacity_factor_lag_24h"] = wall_clock_lag(vre_capacity_factor, 24)

print("lag features built:")
for col in ["residual_load_lag_24h", "residual_load_lag_168h", "vre_capacity_factor_lag_24h"]:
    n_missing = features[col].isna().sum()
    print(f"  {col:32s} missing: {n_missing:,} ({n_missing / len(features):.3%})")

In [ ]:
day_start_mask = time_series["hour"] == 0
day_start_index = time_series.index[day_start_mask]

# closed="left" excludes the midnight row itself, so the window covers strictly the 72h/168h
# BEFORE that day's first hour — never any hour of the day being anchored.
# min_periods = the full window size: an early, partial window is left NaN rather than
# silently computed from fewer observations (pandas time-based rolling defaults to
# min_periods=1, which would otherwise mask the warm-up period).
roll_mean_72h_at_midnight = (
    time_series["residual_load"].rolling("72h", closed="left", min_periods=72).mean().loc[day_start_index]
)
roll_std_168h_at_midnight = (
    time_series["residual_load"].rolling("168h", closed="left", min_periods=168).std().loc[day_start_index]
)

# Broadcast each day's single midnight-anchored value to every hour of that calendar day.
date_to_roll_mean = pd.Series(roll_mean_72h_at_midnight.to_numpy(), index=day_start_index.date)
date_to_roll_std = pd.Series(roll_std_168h_at_midnight.to_numpy(), index=day_start_index.date)

features["residual_load_roll_mean_72h"] = time_series["date"].map(date_to_roll_mean)
features["residual_load_roll_std_168h"] = time_series["date"].map(date_to_roll_std)

print("rolling features built (one value per calendar day, broadcast to its hours):")
for col in ["residual_load_roll_mean_72h", "residual_load_roll_std_168h"]:
    n_missing = features[col].isna().sum()
    n_days_missing = features.loc[features[col].isna()].join(time_series["date"])["date"].nunique()
    print(f"  {col:32s} missing: {n_missing:,} hours across {n_days_missing} calendar days")

**Why more than the initial warm-up is missing.** Beyond the record's first 3 days (72h window) and first 7 days (168h window), each rolling feature is also `NaN` for 3 more days after `residual_load_roll_mean_72h`'s spring DST transition, and 7 more days after `residual_load_roll_std_168h`'s — because the missing spring-DST hour (per CLAUDE.md, one per year) means a trailing 72h/168h window spanning that hour contains one fewer observation than `min_periods` requires, so it is correctly rejected rather than silently computed from 71/167 hours. Concretely: 3 initial + 8 years × 3 = 27 days for the 72h window, and 7 initial + 8 years × 7 = 63 days for the 168h window — exactly what the counts above show. This is the same "never interpolate, reject incomplete windows" discipline the project already applies to day completeness elsewhere, not a defect.

## 4 Forecast-derived features

| Feature | Definition | Why it's origin-safe |
|---|---|---|
| `forecast_wind_solar` | `fc_gen_wind_solar` **for the target day itself** | genuinely published by 18:00 on `D−1` (`04-forecast-metrics.md`), not a same-day actual |
| `forecast_wind_solar_error_lag_24h` | `err_renewables` (from `data/metrics/smard_forecast_errors_hourly.csv`), 24h before `t` | strictly past — same `wall_clock_lag` mechanism as §3, applied to spec 04's own error export rather than re-deriving it |

`forecast_wind_solar` is the **one feature in this notebook allowed to equal the target hour's
own timestamp** — everything else in §2–§3 is either a calendar fact or strictly past. This is
the exception §1.3 names explicitly.

In [ ]:
features["forecast_wind_solar"] = time_series["fc_gen_wind_solar"]

forecast_errors = pd.read_csv(FORECAST_ERRORS, parse_dates=["timestamp"]).set_index("timestamp")
assert forecast_errors.index.equals(time_series.index), (
    "smard_forecast_errors_hourly.csv is not aligned with time_series — re-run "
    "forecast-metrics-claude.ipynb (spec 04) against the current data/smard.csv"
)

features["forecast_wind_solar_error_lag_24h"] = wall_clock_lag(forecast_errors["err_renewables"], 24)

print("forecast-derived features built:")
for col in ["forecast_wind_solar", "forecast_wind_solar_error_lag_24h"]:
    n_missing = features[col].isna().sum()
    print(f"  {col:32s} missing: {n_missing:,} ({n_missing / len(features):.3%})")

**Missingness check.** Per `04-forecast-metrics.md`, `fc_gen_wind_solar` is complete in the
current snapshot, so `forecast_wind_solar` should have (close to) zero missing hours;
`fc_residual_load`/`fc_grid_load` miss one full day (2020-01-31), which does not affect
`err_renewables`. Confirmed below.

## 5 Dropped and excluded candidates

Two candidates were considered while drafting this feature set and dropped; three more columns are explicitly out of scope, not candidates. Both are recorded here rather than left as prose only in the spec, so the decision is not silently lost.

### 5.1 Dropped candidates

| Feature | Definition | Reason dropped |
|---|---|---|
| `wind_solar_lag_24h` | `(wind_off + wind_on + solar)` 24h before target hour | Redundant with `vre_capacity_factor_lag_24h`, which carries the same recent-generation information already normalised by installed capacity — keeping both adds correlated features without new signal. |
| `wind_solar_share` | `(wind_off + wind_on + solar) / grid_load` at the same hour | Requires `grid_load` for the target hour itself, which is not known at origin (no grid-load forecast is in scope) — would be a leakage risk if computed for `D`, and a lagged version would duplicate `vre_capacity_factor_lag_24h`. |

### 5.2 Explicitly excluded (not candidates, not dropped)

| Column | Reason |
|---|---|
| `Forecast Grid Load` (`fc_grid_load`) | Out of scope for this dataset selection; no grid-load forecast is used as a feature. Combined with `Forecast Wind + Solar` it would let a model reconstruct `residual_load` by simple subtraction rather than forecasting it — demonstrated numerically in `data-leakage-demo-claude.ipynb`. |
| `Forecast Residual Load` (`fc_residual_load`) | The benchmark to beat (per CLAUDE.md and `04-forecast-metrics.md`); using it as a feature would make the model dependent on SMARD's own forecast and defeat the comparison. Retained in the pipeline only for evaluation. |
| `Residual Load`, `Grid Load`, `Wind`, `Solar` for the target day itself | These are the outcome (or its components) for `D` — never available at origin, never inputs. |

## 6 Origin-safety self-check

Per Behaviour 7: recompute every lag/rolling/forecast feature independently from its stated
definition and assert it matches what §2–§4 built, plus the structural invariants
(`features` shares `time_series.index`; `time_series` still holds exactly `SERIES + DERIVED`).
This re-derivation is the actual proof of no leakage: each definition below is written from
the timestamp arithmetic alone, independent of the feature-building cells' implementation, so
an accidental same-day read in §2–§4 would show up as a mismatch here.

In [ ]:
# --- Structural invariants ---
assert features.index.equals(time_series.index), "features must share time_series's index"
assert len(features) == len(time_series), "features must have exactly len(time_series) rows"
assert list(time_series.columns) == SERIES + DERIVED, (
    "time_series must still hold exactly SERIES + DERIVED — no feature column leaked onto it"
)

EXPECTED_COLUMNS = [
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "doy_sin", "doy_cos",
    "is_weekend", "is_holiday", "season",
    "residual_load_lag_24h", "residual_load_lag_168h",
    "vre_capacity_factor_lag_24h",
    "residual_load_roll_mean_72h", "residual_load_roll_std_168h",
    "forecast_wind_solar", "forecast_wind_solar_error_lag_24h",
]
assert list(features.columns) == EXPECTED_COLUMNS, list(features.columns)

print(f"structural invariants passed: {len(EXPECTED_COLUMNS)} feature columns, "
      f"{len(features):,} rows, time_series unchanged")

In [ ]:
# --- Calendar / cyclical features: recomputed purely from the target row's own calendar,
# never from a measurement, so they are safe by construction. Recompute independently and
# compare, to catch any accidental copy-paste from the wrong column.
hour = time_series["hour"].to_numpy()
dow = time_series["dow"].to_numpy()
doy = time_series.index.dayofyear.to_numpy()

assert np.allclose(features["hour_sin"], np.sin(2 * np.pi * hour / 24))
assert np.allclose(features["hour_cos"], np.cos(2 * np.pi * hour / 24))
assert np.allclose(features["dow_sin"], np.sin(2 * np.pi * dow / 7))
assert np.allclose(features["dow_cos"], np.cos(2 * np.pi * dow / 7))
assert np.allclose(features["doy_sin"], np.sin(2 * np.pi * doy / 365.25))
assert np.allclose(features["doy_cos"], np.cos(2 * np.pi * doy / 365.25))
assert (features["is_weekend"] == time_series["is_weekend"].astype(int)).all()
assert (features["season"] == time_series["season"]).all()

print("calendar / cyclical features: recomputation matches")

In [ ]:
# --- Lag features: for every non-null row, the feature's timestamp minus its stated lag must
# land exactly on the source timestamp actually used, and that source timestamp must be
# strictly earlier — i.e. a lag can never coincide with or postdate its own target hour.
def assert_exact_lag(feature_col: str, source: pd.Series, hours: int) -> None:
    valid = features[feature_col].notna()
    source_timestamps = features.index[valid] - pd.Timedelta(hours=hours)
    assert (source_timestamps < features.index[valid]).all(), (
        f"{feature_col}: source timestamp is not strictly before the target hour"
    )
    recomputed = source.reindex(source_timestamps).to_numpy()
    assert np.allclose(features.loc[valid, feature_col].to_numpy(), recomputed, equal_nan=True), (
        f"{feature_col}: does not match an independent {hours}h wall-clock lag of its source"
    )
    print(f"{feature_col:36s} matches an independent {hours}h lag "
          f"({valid.sum():,} non-null rows checked)")


assert_exact_lag("residual_load_lag_24h", time_series["residual_load"], 24)
assert_exact_lag("residual_load_lag_168h", time_series["residual_load"], 168)
assert_exact_lag("vre_capacity_factor_lag_24h", vre_capacity_factor, 24)
assert_exact_lag("forecast_wind_solar_error_lag_24h", forecast_errors["err_renewables"], 24)

In [ ]:
# --- Rolling features: every hour of day D must carry the SAME value (one snapshot per
# origin), and that value must equal an independent trailing-window computation ending
# strictly before D's first hour.
def assert_day_anchored_rolling(feature_col: str, source: pd.Series, hours: int) -> None:
    per_day = features.groupby(time_series["date"])[feature_col].nunique(dropna=True)
    assert (per_day <= 1).all(), f"{feature_col}: not constant within every calendar day"

    window_end = pd.DatetimeIndex(sorted(time_series["date"].unique()))
    window_start = window_end - pd.Timedelta(hours=hours)
    assert (window_end < window_end + pd.Timedelta(hours=1)).all()  # window_end IS the day start

    recomputed = {}
    for start, end in zip(window_start, window_end):
        window_values = source.loc[(source.index >= start) & (source.index < end)]
        recomputed[end.date()] = (
            window_values.mean() if len(window_values) >= hours else np.nan
        ) if "mean" in feature_col else (
            window_values.std() if len(window_values) >= hours else np.nan
        )

    actual = features.groupby(time_series["date"])[feature_col].first()
    expected = pd.Series(recomputed)
    assert np.allclose(actual.to_numpy(), expected.reindex(actual.index).to_numpy(), equal_nan=True), (
        f"{feature_col}: does not match an independent {hours}h trailing window ending at day start"
    )
    print(f"{feature_col:36s} constant per day and matches an independent {hours}h window "
          f"({actual.notna().sum():,} days checked)")


assert_day_anchored_rolling("residual_load_roll_mean_72h", time_series["residual_load"], 72)
assert_day_anchored_rolling("residual_load_roll_std_168h", time_series["residual_load"], 168)

In [ ]:
# --- Forecast feature: the ONE column allowed to equal its own target timestamp.
assert (features["forecast_wind_solar"] == time_series["fc_gen_wind_solar"]).all(), (
    "forecast_wind_solar must equal fc_gen_wind_solar at the SAME timestamp — this is the "
    "one declared origin-safety exception (Behaviour 6)"
)
print("forecast_wind_solar matches fc_gen_wind_solar at its own timestamp (declared exception)")

print("\nall origin-safety checks passed")

## 7 Export

`data/features/residual_load_features.csv`: one row per hour of `time_series`, `timestamp`
plus all feature columns. Plain CSV (`sep=","`, `decimal="."`, UTF-8) — the project's
derived-file convention (matching `data/metrics/` and `data/risk_classification/`), not
SMARD's German-Excel format. The notebook creates `data/features/` if it is missing rather
than relying on the `.gitkeep`.

In [ ]:
FEATURES_DIR = DATA_DIR / "features"
FEATURES_DIR.mkdir(exist_ok=True)
OUTPUT_PATH = FEATURES_DIR / "residual_load_features.csv"

export = features.reset_index().rename(columns={"index": "timestamp"})
export.to_csv(OUTPUT_PATH, sep=",", decimal=".", index=False, encoding="utf-8")

print(f"wrote {OUTPUT_PATH}")
print(f"  {export.shape[0]:,} rows x {export.shape[1]} columns")
print(f"  columns: {list(export.columns)}")

### 7.1 Verify the export

Re-read the written file and confirm it round-trips exactly: same shape, same values as
`features` (up to the float precision CSV round-tripping preserves).

In [ ]:
reloaded = pd.read_csv(OUTPUT_PATH, parse_dates=["timestamp"]).set_index("timestamp")

assert reloaded.shape == features.shape, (reloaded.shape, features.shape)
assert reloaded.index.equals(features.index)
assert list(reloaded.columns) == list(features.columns)

numeric_cols = [c for c in features.columns if c != "season"]
assert np.allclose(
    reloaded[numeric_cols].to_numpy(dtype=float),
    features[numeric_cols].to_numpy(dtype=float),
    equal_nan=True,
), "reloaded numeric columns do not match the in-memory features frame"
assert (reloaded["season"].astype(str) == features["season"].astype(str)).all()

print("export round-trips exactly")
print(f"\nfinal feature set: {len(features.columns)} columns, {len(features):,} rows")
reloaded.head(3)